# 02. Caches: LRU, LFU, TTL

A **cache** keeps recently/frequently used data close so that repeated requests are fast. When it is full, an **eviction policy** decides what to remove. Caches appear in CPUs, browsers, databases, CDNs and every backend.

**Learning order:** WHY CACHE → LRU WITH OrderedDict → LFU O(1) → TTL → POLICY COMPARISON → INTERVIEW THINKING

## 1. Eviction policies

| Policy | Evicts | Good when |
|---|---|---|
| **LRU** (least recently used) | the one not used for the longest time | recent items are likely to be reused (most workloads) |
| **LFU** (least frequently used) | the one used the fewest times | some items are popular for a long time |
| **TTL** (time to live) | entries older than a limit | data goes stale (sessions, DNS) |
| FIFO | the oldest inserted | simple, rarely best |

## 2. LRU with `OrderedDict`

In the Linked List topic you built LRU with a hash map + doubly linked list. Python's `OrderedDict` already is that: `move_to_end(key)` marks it recently used, `popitem(last=False)` removes the oldest.

In [1]:
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.data = OrderedDict()

    def get(self, key):
        if key not in self.data:
            return -1
        self.data.move_to_end(key)                     # most recently used goes to the end
        return self.data[key]

    def put(self, key, value):
        if key in self.data:
            self.data.move_to_end(key)
        self.data[key] = value
        if len(self.data) > self.cap:
            self.data.popitem(last=False)              # evict the least recently used

c = LRUCache(2)
c.put(1, 1); c.put(2, 2)
print(c.get(1))   # 1
c.put(3, 3)
print(c.get(2), c.get(3))   # -1 3

1
-1 3


## 3. LFU Cache in O(1) (hard)

Evict the **least frequently used** key; if there is a tie, evict the **least recently used among them**.

Structures:
- `vals[key] = value`, `freq[key] = count`
- `buckets[count]` = an `OrderedDict` of keys with that count (insertion order = recency)
- `min_freq` = the smallest count present (to find the eviction bucket instantly)

**On use of a key:** remove it from `buckets[count]`, add to `buckets[count + 1]`; if `buckets[min_freq]` became empty and it was the used key's bucket, `min_freq += 1`. **On new key:** insert with count 1, `min_freq = 1`.

In [2]:
from collections import defaultdict

class LFUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.vals, self.freq = {}, {}
        self.buckets = defaultdict(OrderedDict)         # count -> OrderedDict of keys
        self.min_freq = 0

    def _touch(self, key):
        f = self.freq[key]
        del self.buckets[f][key]
        if not self.buckets[f] and self.min_freq == f:
            self.min_freq += 1
        self.freq[key] = f + 1
        self.buckets[f + 1][key] = None

    def get(self, key):
        if key not in self.vals:
            return -1
        self._touch(key)
        return self.vals[key]

    def put(self, key, value):
        if self.cap == 0:
            return
        if key in self.vals:
            self.vals[key] = value
            self._touch(key)
            return
        if len(self.vals) >= self.cap:
            evict, _ = self.buckets[self.min_freq].popitem(last=False)   # LRU among the least frequent
            del self.vals[evict]
            del self.freq[evict]
        self.vals[key] = value
        self.freq[key] = 1
        self.buckets[1][key] = None
        self.min_freq = 1

c = LFUCache(2)
c.put(1, 1); c.put(2, 2)
print(c.get(1))   # 1
c.put(3, 3)       # evicts key 2 (frequency 1)
print(c.get(2), c.get(3))   # -1 3
c.put(4, 4)       # keys 1 and 3 both have frequency 2: evicts key 1 (older)
print(c.get(1), c.get(3), c.get(4))   # -1 3 4

1
-1 3
-1 3 4


## 4. TTL cache (entries expire)

Store `(value, expiry_time)`. **Lazy deletion:** check the expiry only when the key is read (and clean up then). For large systems a background sweep or a heap of expiry times removes unused expired entries. Here the clock is a **parameter**, which also makes the code testable.

In [3]:
class TTLCache:
    def __init__(self, ttl, clock):
        self.ttl, self.clock, self.data = ttl, clock, {}

    def put(self, key, value):
        self.data[key] = (value, self.clock() + self.ttl)

    def get(self, key):
        item = self.data.get(key)
        if item is None:
            return None
        value, expires = item
        if self.clock() >= expires:
            del self.data[key]                         # lazy expiry
            return None
        return value

now = [0]
cache = TTLCache(10, lambda: now[0])
cache.put("session", "abc")
print(cache.get("session"))   # abc
now[0] = 10
print(cache.get("session"))   # None

abc
None


## 5. Interview thinking

**Ask:** "What should be evicted? Is it thread-safe? What are the key/value types? Is capacity in items or bytes?"

**Say:** "For O(1) LRU I need a hash map for lookup and an ordered structure for recency: a doubly linked list, or Python's OrderedDict. LFU adds frequency buckets and a `min_freq` pointer."

**Common mistakes (LFU):** not updating `min_freq` correctly, forgetting that `put` of an existing key also counts as a use, capacity 0, evicting before checking whether the key already exists.

**Production note:** real caches also handle concurrency (locks/sharding), memory limits, and cache invalidation (the hard part).

## Quick Revision

- LRU = map + recency order (`OrderedDict.move_to_end`, `popitem(last=False)`).
- LFU = `freq` map + `buckets[count]` (OrderedDict) + `min_freq`.
- TTL = store expiry, delete lazily on access.
- Every `get` (and `put` of an existing key) counts as a use.